In [ ]:
# === Tutorial bootstrap: fetch utils/ + sample data if missing (for Colab blob links) ===
import os, sys, urllib.request

REPO   = "amirfar76/neurips25-valid-hparam-selection"
BRANCH = "main"
BASE   = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"

def ensure_utils():
    os.makedirs("utils", exist_ok=True)
    for fname in ["csvio.py", "testing.py"]:
        url = f"{BASE}/utils/{fname}"
        dst = os.path.join("utils", fname)
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url, dst)
    if "utils" not in sys.path:
        sys.path.append(os.path.abspath("utils"))

def ensure_data():
    os.makedirs("data", exist_ok=True)
    for fname in ["sample_binary_losses.csv", "sample_real_losses.csv"]:
        url = f"{BASE}/data/{fname}"
        dst = os.path.join("data", fname)
        if not os.path.exists(dst):
            urllib.request.urlretrieve(url, dst)

ensure_utils()
ensure_data()
print("Bootstrap done: utils/ and data/ available.")


# A (CSV) — LTT: Average Risk from Loss CSV
Each row is a hyperparameter; columns `loss_1..loss_n` are calibration losses.

**Rules**
- If losses are binary, use exact one-sided Binomial tail.
- Otherwise, use Hoeffding (conservative).

In [ ]:
import numpy as np, pandas as pd
from utils.csvio import load_losses_csv, is_binary_array, summarize
from utils.testing import one_sided_binomial_pval, holm_bonferroni
csv_path = 'data/sample_binary_losses.csv'  # <- change to your file
alpha_risk = 0.2                           # target avg loss threshold
alpha_mtp  = 0.05                          # FWER level


In [ ]:
ids, L, cols = load_losses_csv(csv_path)
m, n = L.shape
rows = []
binary = is_binary_array(L)
for i in range(m):
    losses = L[i]
    rhat = float(losses.mean())
    if binary:
        k = int(losses.sum())
        p = one_sided_binomial_pval(k, len(losses), alpha_risk)
        src = 'binomial'
    else:
        p = float(np.exp(-2*len(losses)*max(0.0, alpha_risk - rhat)**2))
        src = 'hoeffding'
    rows.append({'hyperparam_id': ids[i], 'mean_loss': rhat, 'pval': p, 'method': src})
df = pd.DataFrame(rows)
df['selected'] = holm_bonferroni(df['pval'].values, alpha=alpha_mtp)
df.sort_values(['selected','mean_loss'], ascending=[False, True])

**How to use with your data**
- Put your CSV in `data/`.
- Ensure first column is `hyperparam_id` and losses are `loss_1...loss_n`.
- Adjust `alpha_risk` and rerun.